# People Flow Detection using Object Tracking & Heatmap Visualization

## Overview
This project analyzes a video feed to detect, track, and monitor the movement of people in real-time. It counts how many people **enter (IN)** or **exit (OUT)** a monitored zone by detecting virtual-line crossings, and visualizes cumulative foot-traffic density as a **heatmap**.

| Component | Tool |
|-----------|------|
| Detection | YOLOv8n (COCO class 0 = person) |
| Tracking | ByteTrack via `supervision` |
| Counting | Dual-line crossing logic |
| Heatmap | Gaussian-blurred center-point accumulation (JET colormap) |

---

## Detection Method

- **YOLOv8n (Ultralytics)** — lightweight single-stage object detector trained on COCO. Only class `0` (person) is used, filtering out all other objects.
- **ByteTrack (Roboflow supervision)** — assigns a persistent unique ID to each detected person across frames, even through brief occlusions.

---

## Line Coordinates & IN/OUT Logic

Lines are positioned as a **fraction of the frame height**, chosen using [polygonzone.roboflow.com](https://polygonzone.roboflow.com/).

```
Frame height H = 1080 px  (1920 x 1080 video)

  y =  432  (40% of H)  ──────────────────  Upper Line  [GREEN]  →  IN
  y =  648  (60% of H)  ──────────────────  Lower Line  [RED]    →  OUT
```

### Counting Algorithm

For every tracked person, their previous bounding-box center-y is stored in `track_prev_y`.  
Each frame, `prev_y` (last position) is compared with `cy` (current position):

| Condition | Meaning | Action |
|-----------|---------|--------|
| `prev_y < upper_y` **and** `cy >= upper_y` | Person moved **down** past the upper line | `in_count += 1` |
| `prev_y > lower_y` **and** `cy <= lower_y` | Person moved **up** past the lower line | `out_count += 1` |

Each tracker ID is added to a `counted_in` / `counted_out` set after being counted, so noisy or borderline crossings are never double-counted.

---

## Expected Outputs

| File | Description |
|------|-------------|
| `output_flow_tracking.mp4` | Annotated video: bounding boxes, tracker IDs, colour-coded lines, live IN/OUT counter |
| `final_heatmap.png` | Pure heatmap — JET colormap over black background |
| `final_heatmap_overlay.png` | Heatmap blended onto the last video frame for spatial context |

---

## Dependencies

```
ultralytics        # YOLOv8 detection
supervision        # ByteTrack tracker + annotators
opencv-python      # Frame processing & drawing
numpy              # Heatmap accumulation
tqdm               # Progress bar
matplotlib         # Final visualisation
```

In [ ]:
# Install required packages
%pip install -q ultralytics supervision opencv-python numpy tqdm matplotlib

In [ ]:
import os
import urllib.request
import warnings

import cv2
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from ultralytics import YOLO

# Suppress deprecation warning for ByteTrack (renamed in supervision 0.28)
warnings.filterwarnings("ignore", category=FutureWarning)
import supervision as sv

print(f"supervision version: {sv.__version__}")

## 1. Configuration

Line coordinates were chosen using [polygonzone.roboflow.com](https://polygonzone.roboflow.com/).

- **Upper line (green)** — positioned at 40 % of the frame height. A person whose center crosses this line **moving downward (↓)** is counted as **IN**.
- **Lower line (red)** — positioned at 60 % of the frame height. A person whose center crosses this line **moving upward (↑)** is counted as **OUT**.

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
VIDEO_URL    = "https://media.roboflow.com/supervision/video-examples/people-walking.mp4"
VIDEO_PATH   = "people-walking.mp4"
OUTPUT_VIDEO = "output_flow_tracking.mp4"
OUTPUT_HMAP  = "final_heatmap.png"

# ── Line positions (fraction of frame height) ──────────────────────────────
UPPER_LINE_RATIO = 0.40   # IN  – crossing downward
LOWER_LINE_RATIO = 0.60   # OUT – crossing upward

# ── Heatmap radius painted at each detection centre ──────────────────────
HEATMAP_RADIUS = 20

## 2. Download Video & Load Model

In [ ]:
if not os.path.exists(VIDEO_PATH):
    print("Downloading video...")
    urllib.request.urlretrieve(VIDEO_URL, VIDEO_PATH)
    print("Done.")
else:
    print(f"Video already present: {VIDEO_PATH}")

model      = YOLO("yolov8n.pt")
video_info = sv.VideoInfo.from_video_path(VIDEO_PATH)
W, H       = video_info.width, video_info.height

upper_y = int(H * UPPER_LINE_RATIO)
lower_y = int(H * LOWER_LINE_RATIO)

print(f"Resolution : {W} x {H}  |  Frames: {video_info.total_frames}")
print(f"Upper (IN) : y = {upper_y}")
print(f"Lower (OUT): y = {lower_y}")

## 3. Main Processing Loop

### Counting Logic

For every tracked person we store their **previous center-y** (`track_prev_y`).  
Each frame we compare `prev_y` vs the new `cy`:

```
IN  : prev_y < upper_y  AND  cy >= upper_y   → person moved DOWN past the upper line
OUT : prev_y > lower_y  AND  cy <= lower_y   → person moved UP   past the lower line
```

We use `counted_in / counted_out` sets so each person is counted at most once per direction.

In [ ]:
# ── Tracker & annotators ───────────────────────────────────────────────────
tracker         = sv.ByteTrack()
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

# ── State ──────────────────────────────────────────────────────────────────
track_prev_y: dict = {}          # tracker_id -> last cy
counted_in:  set   = set()       # IDs already counted as IN
counted_out: set   = set()       # IDs already counted as OUT
in_count           = 0
out_count          = 0
heatmap_accum      = np.zeros((H, W), dtype=np.float32)
last_frame         = None

# ── Processing ─────────────────────────────────────────────────────────────
print("Processing frames...")
with sv.VideoSink(OUTPUT_VIDEO, video_info) as sink:
    for frame in tqdm(
        sv.get_video_frames_generator(VIDEO_PATH),
        total=video_info.total_frames
    ):
        # ── Detection ──────────────────────────────────────────────────────
        results    = model(frame, classes=[0], verbose=False)[0]
        detections = sv.Detections.from_ultralytics(results)
        detections = tracker.update_with_detections(detections)

        labels = []
        for i in range(len(detections)):
            xyxy = detections.xyxy[i]
            tid  = int(detections.tracker_id[i])

            cx = int((xyxy[0] + xyxy[2]) / 2)
            cy = int((xyxy[1] + xyxy[3]) / 2)

            labels.append(f"#{tid}")

            # ── Heatmap accumulation ────────────────────────────────────────
            cv2.circle(heatmap_accum, (cx, cy), HEATMAP_RADIUS, 1, thickness=-1)

            # ── Crossing detection ──────────────────────────────────────────
            if tid in track_prev_y:
                prev_y = track_prev_y[tid]

                # Person moved DOWN past the upper line -> IN
                if prev_y < upper_y <= cy and tid not in counted_in:
                    in_count += 1
                    counted_in.add(tid)

                # Person moved UP past the lower line -> OUT
                if prev_y > lower_y >= cy and tid not in counted_out:
                    out_count += 1
                    counted_out.add(tid)

            track_prev_y[tid] = cy

        # ── Draw annotations ────────────────────────────────────────────────
        frame = box_annotator.annotate(scene=frame, detections=detections)
        frame = label_annotator.annotate(scene=frame, detections=detections, labels=labels)

        # Virtual lines (OpenCV putText is ASCII-only, no Unicode arrows)
        cv2.line(frame, (0, upper_y), (W, upper_y), (0, 255, 0), 3)
        cv2.putText(frame, "Upper Line [IN  crossing down]",
                    (10, upper_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 255, 0), 2)

        cv2.line(frame, (0, lower_y), (W, lower_y), (0, 0, 255), 3)
        cv2.putText(frame, "Lower Line [OUT crossing up]",
                    (10, lower_y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2)

        # Live counter
        cv2.rectangle(frame, (10, 10), (270, 125), (0, 0, 0), -1)
        cv2.putText(frame, f"IN : {in_count}",  (20, 65),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 255, 0), 3)
        cv2.putText(frame, f"OUT: {out_count}", (20, 115),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.3, (0, 0, 255), 3)

        sink.write_frame(frame)
        last_frame = frame.copy()

print(f"\nIN : {in_count}  |  OUT: {out_count}")
print(f"Output video saved -> {OUTPUT_VIDEO}")

## 4. Generate Heatmap

In [ ]:
print("Generating heatmap...")

blurred       = cv2.GaussianBlur(heatmap_accum, (99, 99), 0)
normed        = cv2.normalize(blurred, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
heatmap_color = cv2.applyColorMap(normed, cv2.COLORMAP_JET)

# Save standalone heatmap
cv2.imwrite(OUTPUT_HMAP, heatmap_color)

# Save blended overlay on the last video frame
if last_frame is not None:
    overlay = cv2.addWeighted(last_frame, 0.45, heatmap_color, 0.55, 0)
    overlay_path = OUTPUT_HMAP.replace(".png", "_overlay.png")
    cv2.imwrite(overlay_path, overlay)
    print(f"Overlay saved → {overlay_path}")

print(f"Heatmap saved → {OUTPUT_HMAP}")

## 5. Sample Frames from Output Video

Four frames sampled across the output video — shows bounding boxes, tracker IDs, colour-coded lines, and the live IN/OUT counter.

In [ ]:
cap         = cv2.VideoCapture(OUTPUT_VIDEO)
total_f     = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
sample_idxs = [5, total_f // 3, 2 * total_f // 3, total_f - 5]

fig, axes = plt.subplots(1, 4, figsize=(22, 5))
for ax, idx in zip(axes, sample_idxs):
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f"Frame {idx}", fontsize=11)
    ax.axis("off")
cap.release()

plt.suptitle(f"Annotated Output Video — IN: {in_count}  |  OUT: {out_count}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Heatmap Visualisation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

hmap_bgr = cv2.imread(OUTPUT_HMAP)
axes[0].imshow(cv2.cvtColor(hmap_bgr, cv2.COLOR_BGR2RGB))
axes[0].set_title("Cumulative Foot-Traffic Heatmap", fontsize=14)
axes[0].axis("off")

overlay_path = OUTPUT_HMAP.replace(".png", "_overlay.png")
if os.path.exists(overlay_path):
    ov_bgr = cv2.imread(overlay_path)
    axes[1].imshow(cv2.cvtColor(ov_bgr, cv2.COLOR_BGR2RGB))
    axes[1].set_title("Heatmap Overlay on Last Frame", fontsize=14)
    axes[1].axis("off")
else:
    axes[1].axis("off")

plt.suptitle(f"People Flow  |  IN: {in_count}  |  OUT: {out_count}",
             fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("Output.png", bbox_inches="tight", dpi=120)
plt.show()
print("Summary figure saved -> Output.png")

## 7. Play Output Video Inline

`supervision.VideoSink` writes with the `mp4v` codec which browsers cannot play.  
This cell re-encodes to **H.264** with ffmpeg (pre-installed in Colab) and embeds the result directly in the notebook.

In [ ]:
import subprocess
from IPython.display import HTML, display
from base64 import b64encode

H264_VIDEO = "output_display.mp4"

# Re-encode mp4v -> H.264 so the browser can play it
result = subprocess.run(
    [
        "ffmpeg", "-y", "-i", OUTPUT_VIDEO,
        "-vcodec", "libx264", "-crf", "28", "-preset", "fast",
        "-vf", "scale=960:-2",
        "-movflags", "+faststart",
        H264_VIDEO,
    ],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

if result.returncode == 0 and os.path.getsize(H264_VIDEO) > 0:
    mp4_bytes = open(H264_VIDEO, "rb").read()
    data_url  = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    display(HTML(f"""
        <h3 style="font-family:sans-serif;margin-bottom:6px">
            Output Video &mdash; People Flow Detection
            &nbsp;&nbsp;<span style="color:green">IN: {in_count}</span>
            &nbsp;|&nbsp;<span style="color:red">OUT: {out_count}</span>
        </h3>
        <video width="960" controls style="border:2px solid #444;border-radius:6px">
            <source src="{data_url}" type="video/mp4">
        </video>
    """))
else:
    print("ffmpeg re-encoding failed.")
    print(f"In Colab: open the Files panel (left sidebar) -> right-click {OUTPUT_VIDEO} -> Download")